# Computation part
### Assumptions

- Annualization factor = 252 trading days
# start_date = "2020-01-01"
# end_date = "2025-01-01"
# TRADING_DAYS = 252

- Risk-free rate = 3%
- Long-only portfolio
- Portfolio weights sum to 1
- Sample period: Jan 1, 2020 – Jan 1, 2025

In [ ]:
# daily returns
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickers = ["NVDA","MSFT","TSLA","AMZN","V","ABNB","DIS","XOM","COST","FDX"]

prices = yf.download(tickers, start="2020-01-01", end="2025-01-01")["Close"]

returns = prices.pct_change().dropna()
returns.head()

In [ ]:
# annualized income, annualized volatility
annual_returns = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)
cov_matrix = returns.cov() * 252

summary = pd.DataFrame({
    "Annual Return": annual_returns,
    "Annual Volatility": annual_volatility
})

summary

In [ ]:
# find biggest sharpe ratio
rf = 0.03  # risk-free rate = 3% # Assumption: Fixed annual risk - free rate

num_portfolios = 10000

portfolio_returns = []
portfolio_volatilities = []
sharpe_ratios = []
weights_list = []

for i in range(num_portfolios):
    weights = np.random.random(len(tickers))
    weights = weights / np.sum(weights)          # All weights are non-negative and sum to one (long-only portfolio)

    port_return = np.dot(weights, annual_returns)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = (port_return - rf) / port_volatility

    portfolio_returns.append(port_return)
    portfolio_volatilities.append(port_volatility)
    sharpe_ratios.append(sharpe)
    weights_list.append(weights)

In [ ]:
# find best combination
max_index = np.argmax(sharpe_ratios)

best_return = portfolio_returns[max_index]
best_volatility = portfolio_volatilities[max_index]
best_sharpe = sharpe_ratios[max_index]
best_weights = weights_list[max_index]

print("Best Portfolio Return:", best_return)
print("Best Portfolio Volatility:", best_volatility)
print("Best Sharpe Ratio:", best_sharpe)

optimal_weights = pd.DataFrame({
    "Ticker": tickers,
    "Weight": best_weights
})

optimal_weights = optimal_weights.sort_values (
    by="Weight",
    ascending=False
)

In [ ]:
# efficent frontier
plt.figure(figsize=(10,6))

plt.scatter(portfolio_volatilities, portfolio_returns, c=sharpe_ratios)
plt.colorbar(label="Sharpe Ratio")

plt.scatter(best_volatility, best_return, marker="*", s=300, label="Max Sharpe Portfolio")

plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.title("Efficient Frontier")
plt.legend()
plt.show()

We generated 10,000 random portfolios using Monte Carlo simulation.

Each portfolio's annualized return, volatility, and Sharpe ratio were calculated.

The portfolio with the highest Sharpe ratio was selected as the optimal portfolio because it provides the best return per unit of risk.